# LCEL과 Runnable — 강사용 정답본

원본: `src/solution/Chapter01/02-langchain/01_lcel_runnable.py`  
대응 슬라이드: `docs/raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf`

**학습 목표:** 프롬프트·모델·파서 → 병렬·통과·함수 래핑의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 실행되는 체인이 LLM을 여러 번 호출한다. 모델 설정을 확인하고 각 예시의 호출 횟수를 살펴본다.



## 흐름을 먼저 읽기

프롬프트·모델·파서 → 병렬·통과·함수 래핑

핵심은 각 Runnable의 입력·출력과 반복 흐름이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: 프롬프트·모델·파서 → 병렬·통과·함수 래핑. RunnablePassthrough는 입력의 무엇을 보존하나? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 실행되는 체인이 LLM을 여러 번 호출한다. 모델 설정을 확인하고 각 예시의 호출 횟수를 살펴본다.



In [ ]:
"""
LCEL Runnable: LangChain 구성 요소를 파이프라인으로 연결하기

핵심 개념:
  - Runnable = invoke, stream, batch를 공통으로 제공하는 실행 단위
  - LCEL = prompt → model → parser 흐름을 | 연산자로 선언하는 방식
  - RunnableParallel, RunnablePassthrough, RunnableLambda로 입력 흐름을 조합 가능

실행:
  python 02-langchain/01_lcel_runnable.py
"""
import os

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

from config.llm import get_llm


load_dotenv()

model = get_llm()

# ── 기본 LCEL 체인 ────────────────────────────────────────────────────────────
# ChatPromptTemplate: `{}` 변수만 바꿔 재사용할 수 있는 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 초보자를 위한 AI 튜터야.'),
    ('human', '{topic}을 초급자에게 3문장으로 설명해줘.'),
])

# StrOutputParser: LLM 출력을 문자열로 변환
parser = StrOutputParser()

# LCEL: LangChain Expression Language
# - 프롬프트 작성 → LLM 호출 → 출력 파싱 과정을 하나의 실행 단위로 묶는다.
# - prompt, model, parser는 모두 Runnable 계열 객체다.
# - Runnable | Runnable 결과도 Runnable이므로 체인을 계속 이어붙일 수 있다.
chain = prompt | model | parser

# Runnable 실행 방식:
# - invoke(): 단일 입력 처리
# - stream(): 토큰/청크 단위 출력
# - batch(): 여러 입력 일괄 처리
answer = chain.invoke({'topic': 'LLM'})
print(answer)
print('----')
for chunk in chain.stream({'topic': 'LangChain'}):
    print(chunk, end='', flush=True)
print()

print('----')
inputs = [{'topic': 'LangChain Expression Language'}, {'topic': 'LangChain Runnable'}]
responses = chain.batch(inputs)
for topic_input, response in zip(inputs, responses):
    print(f"{topic_input.get('topic')}: {response.strip()}")


# ── RunnableParallel ──────────────────────────────────────────────────────────
# 여러 Runnable을 병렬로 실행해 결과를 dict로 모을 때 사용
pros_chain = ChatPromptTemplate.from_template('{country}로 여행갈 때의 장점을, 5개 정도 목록 형식으로 설명해줘.') | model | StrOutputParser()
cons_chain = ChatPromptTemplate.from_template('{country}로 여행갈 때의 단점을, 5개 정도 목록 형식으로 설명해줘.') | model | StrOutputParser()

runnable = RunnableParallel(pros=pros_chain, cons=cons_chain)
answers = runnable.invoke({'country': '베트남'})
print('pros')
print(answers.get('pros'))

print('cons')
print(answers.get('cons'))

# .stream() 예시
# answers = {'pros': '', 'cons': ''}
# for chunk in runnable.stream({'country': 'USA'}):
#     for key in chunk:
#         answers[key] += chunk[key]

# from pprint import pprint
# pprint(answers)

# Python dict가 pipe 사이에 들어가면 자동으로 RunnableParallel처럼 동작한다.
parallel_dict = RunnablePassthrough() | {
    'pros': ChatPromptTemplate.from_template('{country}로 여행갈 때의 장점을, 5개 정도 목록 형식으로 설명해줘.') | model | StrOutputParser(),
    'cons': ChatPromptTemplate.from_template('{country}로 여행갈 때의 단점을, 5개 정도 목록 형식으로 설명해줘.') | model | StrOutputParser(),
}

answers = parallel_dict.invoke({'country': '베트남'})
print('pros')
print(answers.get('pros'))

print('cons')
print(answers.get('cons'))


# ── RunnablePassthrough ───────────────────────────────────────────────────────
# 입력값을 그대로 다음 단계에 전달하면서 부가 필드를 추가할 때 사용
def get_question_length(text: str):
    return len(text)

# 예: 사용자 질문을 바탕으로 맥락 정보나 메타데이터를 추가
passthrough = RunnablePassthrough.assign(length=lambda x: get_question_length(x['question']))
print(passthrough.invoke({'question': 'RunnablePassthrough의 역할이 뭐야?'}))

prompt = ChatPromptTemplate.from_template('{question} 3문장으로 설명해줘.')
chain = passthrough | prompt | model | StrOutputParser()
print(chain.invoke({'question': 'RunnablePassthrough의 역할이 뭐야?'}))


# ── RunnableLambda ────────────────────────────────────────────────────────────
# 일반 Python 함수를 Runnable로 감싸 체인 안에서 사용할 때 활용
acronym_dict = {
    'GPT': 'Generative Pre-trained Transformer',
    'LLM': 'Large Language Model',
    'LCEL': 'LangChain Expression Language',
}
def translate_acronym(acronym: str):
    return acronym_dict.get(acronym, acronym)

print(RunnableLambda(translate_acronym).invoke('GPT'))

prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 초보자를 위한 AI 튜터야.'),
    ('human', '{topic}을 초급자에게 3문장으로 설명해줘.'),
])

chain = {
    'topic': RunnableLambda(translate_acronym),
} | prompt | model | StrOutputParser()
print(chain.invoke('GPT'))


## 결과를 검토하기

- 예상 동작: 체인·병렬·입력 통과·함수 래핑의 결과를 서로 다른 형태로 볼 수 있다.
- 확인 질문: RunnablePassthrough는 입력의 무엇을 보존하나?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
